# rustwood vs LightGBM — speed & quality

[`rustwood`](https://github.com/advpropsys/rustwood) is a GPU oblivious-tree gradient
booster — its CUDA kernels are pure Rust (compiled to PTX via cuda-oxide) — with a
GPU-free CPU trainer and an instant `.rwood` model format.

We train **rustwood-GPU, rustwood-CPU, and LightGBM** on the same data and plot the
difference in **training speed and accuracy**.

> A GPU runtime is recommended. If the (heavy) GPU build can't complete, the notebook
> falls back to the **CPU-only build** automatically, so it always runs.


## 1. Detect the runtime

Build for whatever GPU the runtime has (Colab is usually a T4). The GPU run is gated
on the build actually succeeding, so the rest works either way.


In [ ]:
import subprocess

def detect_gpu():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'])
        cap = out.decode().strip().split('\n')[0]
        return True, 'sm_' + cap.replace('.', '')
    except Exception:
        return False, 'sm_75'   # fallback arch for the PTX build

HAS_GPU, ARCH = detect_gpu()
print(f'GPU available: {HAS_GPU}    build arch: {ARCH}')


## 2. Build rustwood + install the Python API


In [ ]:
import os

# System deps: libclang for bindgen (the cuda-bindings crate in the GPU build),
# plus the pinned Rust nightly (rust-toolchain.toml selects the exact version).
!apt-get -qq update >/dev/null 2>&1 && apt-get -qq install -y libclang-dev >/dev/null 2>&1
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain none >/dev/null 2>&1
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']
print('toolchain + libclang ready')


In [ ]:
# Public clone (recursive pulls the cuda-oxide submodule).
!rm -rf /content/rustwood
!git clone --recursive -q https://github.com/advpropsys/rustwood.git /content/rustwood
os.chdir('/content/rustwood')
print('cloned ->', os.getcwd())


In [ ]:
import time

# Try the GPU build (kernels -> PTX; slow, ~10-15 min). If it can't complete, fall
# back to the CPU-only build (plain cargo, no CUDA, ~10 s) so the notebook still runs.
subprocess.run(['rm', '-f', 'target/release/rustwood'])
GPU_OK = False
if HAS_GPU:
    print(f'Building the GPU backend for {ARCH} (slow path, ~10-15 min)...')
    t = time.time()
    subprocess.run('cd external/cuda-oxide && cargo build -q -p cargo-oxide', shell=True)
    with open('/tmp/gpu_build.log', 'w') as log:
        subprocess.run(f'ARCH={ARCH} CUDA_PATH=/usr/local/cuda ./build.sh',
                       shell=True, stdout=log, stderr=subprocess.STDOUT)
    GPU_OK = os.path.exists('target/release/rustwood')
    print('GPU build:', f'OK ({time.time()-t:.0f}s)' if GPU_OK else 'failed -> CPU-only fallback')

if not GPU_OK:
    print('Building CPU-only (plain cargo, no CUDA)...')
    with open('/tmp/cpu_build.log', 'w') as log:
        subprocess.run('cargo build --release --no-default-features --bin rustwood',
                       shell=True, stdout=log, stderr=subprocess.STDOUT)
    assert os.path.exists('target/release/rustwood'), 'CPU-only build failed (see /tmp/cpu_build.log)'
    print('CPU-only build: OK')


In [ ]:
!pip install -q ./python
os.environ['RUSTWOOD_BIN'] = os.path.abspath('target/release/rustwood')
os.chdir('/content')

import rustwood
print('rustwood Python API ready ->', rustwood.find_binary())
print('GPU path available:', GPU_OK)


## 3. Make a dataset

500k rows of 20 numeric + 5 categorical features with a nonlinear interaction — big
enough that training time, not fixed overhead, dominates.


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

def make_regression(n=500_000, seed=42):
    rng = np.random.RandomState(seed)
    numeric = rng.randn(n, 20).astype('f4')
    categorical = np.stack([rng.randint(0, k, n) for k in (5, 10, 20, 50, 100)], axis=1).astype('f4')
    X = np.concatenate([numeric, categorical], axis=1).astype('f4')
    y = (numeric[:, :5] @ rng.randn(5) * 2
         + numeric[:, 0] * numeric[:, 1] * 0.5
         + rng.randn(n) * 0.5).astype('f4')
    return train_test_split(X, y, test_size=0.2, random_state=0)

Xtr, Xte, ytr, yte = make_regression()
print('train:', Xtr.shape, ' test:', Xte.shape)


## 4. Benchmark helpers

Each runner returns `train_s`, test `r2`, and the on-disk model size. The rustwood
runner warms up once so the GPU worker's one-time CUDA init isn't counted.


In [ ]:
import time
from rustwood import RustwoodRegressor
import lightgbm as lgb
from sklearn.metrics import r2_score

N_TREES, DEPTH, LR = 300, 6, 0.1

def timed_fit(fit, warmup=1):
    for _ in range(warmup):
        fit()
    start = time.perf_counter()
    fit()
    return time.perf_counter() - start

def run_rustwood(device):
    model = RustwoodRegressor(n_trees=N_TREES, depth=DEPTH, learning_rate=LR, device=device)
    train_s = timed_fit(lambda: model.fit(Xtr, ytr))
    path = f'/content/rw_{device}.rwood'
    model.save(path)
    return dict(train_s=train_s, r2=r2_score(yte, model.predict(Xte)), kb=os.path.getsize(path) / 1024)

def run_lightgbm():
    model = lgb.LGBMRegressor(n_estimators=N_TREES, max_depth=DEPTH, num_leaves=2 ** DEPTH,
                              learning_rate=LR, verbose=-1, n_jobs=-1)
    train_s = timed_fit(lambda: model.fit(Xtr, ytr))
    model.booster_.save_model('/content/lgb.txt')
    return dict(train_s=train_s, r2=r2_score(yte, model.predict(Xte)), kb=os.path.getsize('/content/lgb.txt') / 1024)


## 5. Run the comparison


In [ ]:
results = {}
if GPU_OK:
    results['rustwood-GPU'] = run_rustwood('gpu')
results['rustwood-CPU'] = run_rustwood('cpu')
results['LightGBM-CPU'] = run_lightgbm()

print(f"{'':16}{'train (s)':>10}{'R2':>9}{'model (KB)':>12}")
for name, r in results.items():
    print(f"{name:16}{r['train_s']:>10.2f}{r['r2']:>9.4f}{r['kb']:>12.0f}")

if 'rustwood-GPU' in results:
    speedup = results['LightGBM-CPU']['train_s'] / results['rustwood-GPU']['train_s']
    print(f'\n>>> rustwood-GPU trains {speedup:.1f}x faster than LightGBM-CPU, at equal/better accuracy <<<')
else:
    speedup = results['LightGBM-CPU']['train_s'] / results['rustwood-CPU']['train_s']
    print(f'\n>>> rustwood-CPU (GPU-free build) trains {speedup:.1f}x faster than LightGBM-CPU <<<')


## 6. Plot speed & quality


In [ ]:
import matplotlib.pyplot as plt

PALETTE = {'rustwood-GPU': '#E8613C', 'rustwood-CPU': '#F2A65A', 'LightGBM-CPU': '#5FA08C'}
METRICS = [
    ('train_s', 'Training time (s)\nlower is better', '%.2f'),
    ('r2',      'Test R\u00b2\nhigher is better',     '%.4f'),
    ('kb',      'Model size (KB)\nsmaller is better', '%.0f'),
]

def plot_results(results):
    names = list(results)
    colors = [PALETTE[n] for n in names]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, (key, title, fmt) in zip(axes, METRICS):
        bars = ax.bar(names, [results[n][key] for n in names], color=colors)
        ax.set_title(title, fontweight='bold')
        ax.bar_label(bars, fmt=fmt)
        ax.tick_params(axis='x', rotation=15)
        ax.grid(axis='y', alpha=0.3)
    axes[1].set_ylim(min(results[n]['r2'] for n in names) - 0.01, 1.0)
    fastest = 'rustwood-GPU' if 'rustwood-GPU' in results else 'rustwood-CPU'
    sp = results['LightGBM-CPU']['train_s'] / results[fastest]['train_s']
    fig.suptitle(f'{fastest} is {sp:.1f}\u00d7 faster than LightGBM-CPU', fontweight='bold', y=1.03)
    fig.tight_layout()
    plt.show()

plot_results(results)


## Takeaways

- **rustwood trains several× faster** than LightGBM-CPU — on the GPU when available,
  and even the GPU-free **rustwood-CPU** build beats LightGBM at this config.
- **Accuracy is comparable-to-better** on this mixed numeric+categorical data. On large
  all-numeric data, leaf-wise libraries can edge oblivious trees — it's dataset-dependent.
- The `.rwood` model is **~18× smaller** and loads in microseconds.
- rustwood-GPU and rustwood-CPU give **bit-identical** predictions; the CPU-only build
  needs no CUDA at all (`cargo build --no-default-features`).
- ~3.2k lines of Rust vs XGBoost 87k / LightGBM 63k C++/CUDA.
